# 02 기준선 A — 사전 학습 wav2vec2-base 특징 고정 + 마지막 층만 학습

- 사전 학습 모델: `torchaudio.pipelines.WAV2VEC2_BASE` (LibriSpeech 960시간 영어 음성으로 자기 지도 학습, 파라미터 약 9,437만 개). 단어 라벨로는 학습된 적 없음
- 방법: wav2vec2는 고정(학습 안 함). 녹음마다 트랜스포머 12개 층의 출력을 시간 방향으로 평균 내 768차원 특징을 한 번만 뽑아 저장 → 그 위에 `Linear(768 → 35)`만 학습
- E02 = 마지막 층(12층) 특징. 층별 비교(E03)는 1~12층 각각으로 같은 Linear를 학습해 어느 층에 단어 정보가 많은지 봄
- 설정과 모델 선택은 검증 묶음으로만 합니다. 시험 묶음은 이 노트북에서 쓰지 않습니다
- Colab: 런타임 유형 T4 GPU → 모두 실행. 처음 실행은 데이터 내려받기(약 2.4GB)와 특징 뽑기 때문에 오래 걸리고, 특징은 `cache/`에 저장해 다음부터는 건너뜁니다
- `SMOKE = True`로 두면 단어마다 몇 개만 써서 1에폭으로 끝까지 도는지만 확인합니다 (GPU가 없을 때)

In [ ]:
# 셀 1. 설정과 버전
import os, sys, time, random, collections
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import soundfile as sf
import matplotlib.pyplot as plt

SMOKE = False                  # True: 단어마다 학습 20개·검증 5개, 1에폭 (코드 확인용)
SEED = 0
DATA_ROOT = "data"             # Colab: /content/data
CACHE_DIR = "cache"            # 뽑은 특징 저장 위치 (Drive를 연결했다면 Drive 경로로 바꾸면 런타임이 끊겨도 남음)
SR = 16000
BATCH_EXTRACT = 64             # 특징 뽑을 때 배치
EPOCHS = 30                    # Linear 학습 에폭
LR = 1e-3
BATCH_TRAIN = 512

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if SMOKE:
    EPOCHS = 1
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchaudio", torchaudio.__version__, "| soundfile", sf.__version__)
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "GPU 없음 → CPU", "| SMOKE", SMOKE)

In [ ]:
# 셀 2. 데이터와 공식 분할 (01 노트북과 같음, 시험 묶음은 불러오지 않음)
os.makedirs(DATA_ROOT, exist_ok=True)
splits = {s: torchaudio.datasets.SPEECHCOMMANDS(DATA_ROOT, download=True, subset=sub)
          for s, sub in [("train", "training"), ("val", "validation")]}
BASE_DIR = os.path.dirname(splits["train"]._path)

items = {s: [] for s in splits}                       # 분할마다 (경로, 단어)
for s, ds in splits.items():
    for i in range(len(ds)):
        path, _, word, _, _ = ds.get_metadata(i)
        items[s].append((path, word))
WORDS = sorted({w for p, w in items["train"]})
W2I = {w: i for i, w in enumerate(WORDS)}

if SMOKE:                                             # 단어마다 앞에서부터 몇 개만
    for s, k in [("train", 20), ("val", 5)]:
        per = collections.defaultdict(list)
        for it in items[s]:
            per[it[1]].append(it)
        items[s] = [it for w in WORDS for it in per[w][:k]]
print({s: len(v) for s, v in items.items()}, "| 단어", len(WORDS))

In [ ]:
# 셀 3. 사전 학습 wav2vec2-base 불러오기 (고정)
bundle = torchaudio.pipelines.WAV2VEC2_BASE
w2v = bundle.get_model().to(device).eval()
for p in w2v.parameters():
    p.requires_grad = False                           # 특징 추출부는 학습하지 않음
n_w2v = sum(p.numel() for p in w2v.parameters())
with torch.inference_mode():
    feats, _ = w2v.extract_features(torch.zeros(1, SR, device=device))
print(f"wav2vec2-base 파라미터 {n_w2v:,}개 (고정) | 층 {len(feats)}개, 1초 → 층마다 {tuple(feats[-1].shape[1:])} (시간 칸, 차원)")

In [ ]:
# 셀 4. 특징 뽑기: 녹음마다 12개 층 출력을 시간 평균 → (12, 768), 한 번만 뽑아 cache에 저장
class Wavs(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, word = self.items[i]
        x, sr = sf.read(os.path.join(BASE_DIR, path), dtype="float32")
        x = torch.from_numpy(x)
        x = torch.nn.functional.pad(x, (0, max(0, SR - len(x))))[:SR]     # 1초로 맞춤
        return x, W2I[word]

def extract(split):
    path = os.path.join(CACHE_DIR, f"w2v_base_{split}{'_smoke' if SMOKE else ''}.pt")
    if os.path.exists(path):
        d = torch.load(path)
        print(f"{split}: 저장된 특징 사용 {tuple(d['X'].shape)} (처음 뽑을 때 {d['sec']:.0f}초)")
        return d
    loader = torch.utils.data.DataLoader(Wavs(items[split]), batch_size=BATCH_EXTRACT, shuffle=False,
                                         num_workers=2 if os.name != "nt" else 0)
    X, Y = [], []
    t0 = time.time()
    with torch.inference_mode(), torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        for b, (x, y) in enumerate(loader):
            layers, _ = w2v.extract_features(x.to(device))
            X.append(torch.stack([h.mean(dim=1) for h in layers], dim=1).half().cpu())   # (B, 12, 768)
            Y.append(y)
            if b % 200 == 0:
                print(f"  {split} {b * BATCH_EXTRACT:>6,}/{len(loader.dataset):,}  {time.time() - t0:.0f}초")
    if device.type == "cuda":
        torch.cuda.synchronize()
    d = {"X": torch.cat(X), "Y": torch.cat(Y), "sec": time.time() - t0}
    os.makedirs(CACHE_DIR, exist_ok=True)
    torch.save(d, path)
    print(f"{split}: {tuple(d['X'].shape)}를 {d['sec']:.0f}초에 뽑음 ({len(d['Y']) / d['sec']:.0f}개/초) → {path}")
    return d

data = {s: extract(s) for s in ["train", "val"]}
EXTRACT_SEC = sum(d["sec"] for d in data.values())

In [ ]:
# 셀 5. 평가 지표와 Linear 학습 함수
def metrics(pred, y, k=len(WORDS)):
    """정확도, 매크로 F1, 혼동 행렬 (행 = 정답, 열 = 예측)"""
    cm = torch.bincount(y * k + pred, minlength=k * k).reshape(k, k).float()
    tp = cm.diag()
    prec = tp / cm.sum(0).clamp(min=1)
    rec = tp / cm.sum(1).clamp(min=1)
    f1 = 2 * prec * rec / (prec + rec).clamp(min=1e-12)
    return (tp.sum() / cm.sum()).item(), f1.mean().item(), cm

def train_linear(layer, epochs=EPOCHS, seed=SEED, log=False):
    """layer번째 층(0~11) 특징 위에 Linear(768 → 35)만 학습. 검증 정확도가 가장 높은 에폭의 결과를 돌려줌"""
    torch.manual_seed(seed)
    Xtr = data["train"]["X"][:, layer].float().to(device); Ytr = data["train"]["Y"].to(device)
    Xva = data["val"]["X"][:, layer].float().to(device);   Yva = data["val"]["Y"].to(device)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6              # 학습 묶음 통계로 표준화
    Xtr, Xva = (Xtr - mu) / sd, (Xva - mu) / sd
    head = nn.Linear(Xtr.shape[1], len(WORDS)).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=LR)
    hist, best = [], None
    t0 = time.time()
    for ep in range(epochs):
        head.train()
        perm = torch.randperm(len(Xtr), device=device)
        tot = 0.0
        for i in range(0, len(Xtr), BATCH_TRAIN):
            idx = perm[i:i + BATCH_TRAIN]
            loss = nn.functional.cross_entropy(head(Xtr[idx]), Ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        head.eval()
        with torch.no_grad():
            acc, f1, cm = metrics(head(Xva).argmax(1), Yva)
        hist.append((tot / len(Xtr), acc, f1))
        if best is None or acc > best["acc"]:
            best = {"acc": acc, "f1": f1, "epoch": ep + 1, "cm": cm.cpu(), "state": {k: v.cpu() for k, v in head.state_dict().items()}}
        if log:
            print(f"  epoch {ep + 1:2d}  train loss {tot / len(Xtr):.4f} | val acc {acc:.4f} | val macro F1 {f1:.4f}")
    best["sec"] = time.time() - t0
    best["hist"] = hist
    best["params"] = sum(p.numel() for p in head.parameters())
    return best

In [ ]:
# 셀 6. E02 기준선 A — 마지막 층(12층) 특징 + Linear
print(f"Linear(768 → {len(WORDS)}) 파라미터: 768 × {len(WORDS)} + {len(WORDS)} = {768 * len(WORDS) + len(WORDS):,}")
A = train_linear(layer=11, log=True)
print(f"E02 기준선 A: 검증 정확도 {A['acc']:.2%}, 매크로 F1 {A['f1']:.4f} (최고 에폭 {A['epoch']}/{EPOCHS}) | Linear 학습 {A['sec']:.1f}초 + 특징 뽑기 {EXTRACT_SEC:.0f}초")

h = np.array(A["hist"])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(range(1, len(h) + 1), h[:, 0], "o-"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("train loss"); ax[0].grid(alpha=.3)
ax[1].plot(range(1, len(h) + 1), h[:, 1], "o-", label="val acc"); ax[1].plot(range(1, len(h) + 1), h[:, 2], "s--", label="val macro F1")
ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.suptitle("E02 baseline A: wav2vec2-base layer 12 (frozen) + Linear"); plt.tight_layout(); plt.show()

In [ ]:
# 셀 7. E03 층별 비교 — 1~12층 특징 각각에 같은 Linear를 학습 (검증 묶음 기준)
layer_res = []
for L in range(12):
    r = train_linear(layer=L)
    layer_res.append(r)
    print(f"  {L + 1:2d}층: 검증 정확도 {r['acc']:.2%}, 매크로 F1 {r['f1']:.4f} ({r['sec']:.1f}초)")
best_L = int(np.argmax([r["acc"] for r in layer_res]))
print(f"검증 정확도가 가장 높은 층: {best_L + 1}층 ({layer_res[best_L]['acc']:.2%}) | 마지막 12층: {layer_res[11]['acc']:.2%}")

plt.figure(figsize=(7, 3))
plt.plot(range(1, 13), [r["acc"] for r in layer_res], "o-", label="val acc")
plt.plot(range(1, 13), [r["f1"] for r in layer_res], "s--", label="val macro F1")
plt.xticks(range(1, 13)); plt.xlabel("wav2vec2 transformer layer"); plt.ylim(0, 1); plt.grid(alpha=.3); plt.legend()
plt.title("E03 which layer holds word information (frozen, Linear probe)"); plt.tight_layout(); plt.show()

In [ ]:
# 셀 8. E02에서 가장 많이 헷갈린 단어 쌍 (오류 분석 준비)
cm = A["cm"].clone()
cm.fill_diagonal_(0)
top = torch.topk(cm.flatten(), 10)
print("정답 → 예측 (검증에서 틀린 횟수)")
for v, i in zip(top.values.tolist(), top.indices.tolist()):
    t, p = divmod(i, len(WORDS))
    print(f"  {WORDS[t]:>9s} → {WORDS[p]:<9s} {int(v)}")
per_word = (A["cm"].diag() / A["cm"].sum(1)).tolist()
order = np.argsort(per_word)
print("단어별 정확도 낮은 5개:", ", ".join(f"{WORDS[i]} {per_word[i]:.1%}" for i in order[:5]))
print("단어별 정확도 높은 5개:", ", ".join(f"{WORDS[i]} {per_word[i]:.1%}" for i in order[::-1][:5]))

In [ ]:
# 셀 9. 실험 결과 표 (이 노트북의 실험 + 01 노트북의 0번 기준)
rows = [
    ("E01", "0번 기준: 학습 묶음에서 가장 많은 단어(zero)만 고름", "0.0385", "0.0021", "-", "01 노트북 출력"),
    ("E02", "기준선 A: wav2vec2-base 12층(마지막) 고정 + Linear", f"{A['acc']:.4f}", f"{A['f1']:.4f}",
     f"특징 {EXTRACT_SEC:.0f}초 + 학습 {A['sec']:.1f}초", f"Linear {A['params']:,}개 학습 / wav2vec2 {n_w2v:,}개 고정"),
    ("E03", f"E02에서 특징 층만 바꿈: 최고 {best_L + 1}층", f"{layer_res[best_L]['acc']:.4f}", f"{layer_res[best_L]['f1']:.4f}",
     f"층마다 학습 약 {np.mean([r['sec'] for r in layer_res]):.1f}초", "1~12층 그래프는 셀 7"),
]
print(f"{'번호':<4} | {'바꾼 설정':<44} | {'검증 정확도':>8} | {'매크로 F1':>8} | {'시간':<22} | 비고")
for r in rows:
    print(" | ".join([r[0].ljust(4), r[1].ljust(44), r[2].rjust(10), r[3].rjust(9), r[4].ljust(24), r[5]]))
print("SMOKE 모드 결과" if SMOKE else f"device {device}, EPOCHS {EPOCHS}, LR {LR}, 시드 {SEED}")